In [1]:
def visualize_directions_improved(df, cluster_angles, cluster_to_name, file_path, video_path=None):
    """
    Enhanced visualization of vehicle trajectories that clearly shows the direction patterns
    overlaid on the video frame with improved visual clarity.
    """
    import os
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from collections import Counter
    from matplotlib.lines import Line2D
    
    if cluster_angles is None or cluster_to_name is None:
        return
    
    # Create output filename
    base_filename = file_path.rsplit('.', 1)[0]
    video_id = os.path.splitext(os.path.basename(file_path))[0]
    video_id = video_id.replace("_manual", "")
    
    # Get first frame of video
    first_frame = None
    if video_path and os.path.exists(video_path):
        try:
            cap = cv2.VideoCapture(video_path)
            if cap.isOpened():
                ret, first_frame = cap.read()
                if not ret:
                    print("Could not read frame from video")
                    first_frame = None
                cap.release()
        except Exception as e:
            print(f"Error reading video: {e}")
    
    # Create a blank canvas if no frame is available
    if first_frame is None:
        first_frame = np.ones((720, 1280, 3), dtype=np.uint8) * 255
        print("Using blank canvas")
    
    # Get frame dimensions
    frame_height, frame_width = first_frame.shape[:2]
    
    # Create colors for each direction cluster
    colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_to_name)))
    direction_to_color = {}
    
    for i, cluster_idx in enumerate(cluster_to_name.keys()):
        # Convert to OpenCV BGR format
        rgb = colors[i][:3]  # RGB components
        bgr = (int(rgb[2]*255), int(rgb[1]*255), int(rgb[0]*255))  # Convert to BGR
        direction_to_color[cluster_to_name[cluster_idx]] = bgr
    
    # Create a copy of the first frame for drawing
    visualization = first_frame.copy()
    
    # Calculate the center point of the frame
    center_x, center_y = frame_width // 2, frame_height // 2
    
    # Calculate appropriate scaling for trajectories to fit the frame
    # Get all trajectory coordinates to find bounds
    all_x_coords = []
    all_y_coords = []
    for _, row in df.iterrows():
        points = row.get('trajectory_points', [])
        if len(points) >= 2:
            x_coords, y_coords = zip(*points)
            all_x_coords.extend(x_coords)
            all_y_coords.extend(y_coords)
    
    # If we have trajectory data, scale appropriately
    if all_x_coords and all_y_coords:
        min_x, max_x = min(all_x_coords), max(all_x_coords)
        min_y, max_y = min(all_y_coords), max(all_y_coords)
        
        # Calculate scaling to fit trajectory data to frame
        # Preserve some margin (10% of each dimension)
        margin_x = frame_width * 0.1
        margin_y = frame_height * 0.1
        
        # Available space for trajectories
        avail_width = frame_width - 2 * margin_x
        avail_height = frame_height - 2 * margin_y
        
        # Calculate scaling factors
        scale_x = avail_width / (max_x - min_x) if max_x > min_x else 1
        scale_y = avail_height / (max_y - min_y) if max_y > min_y else 1
        
        # Use the smaller scaling factor to maintain aspect ratio
        scale = min(scale_x, scale_y) * 0.9  # Extra safety factor
        
        # Calculate offset to center trajectories in frame
        offset_x = margin_x + (avail_width - scale * (max_x - min_x)) / 2
        offset_y = margin_y + (avail_height - scale * (max_y - min_y)) / 2
        
        # Function to scale point coordinates
        def scale_point(point):
            x, y = point
            scaled_x = int(offset_x + (x - min_x) * scale)
            scaled_y = int(offset_y + (y - min_y) * scale)
            return (scaled_x, scaled_y)
        
        # Draw each trajectory with improved visualization
        for _, row in df.iterrows():
            direction = row.get('direction', 'Unknown')
            points = row.get('trajectory_points', [])
            
            if len(points) >= 2:
                # Scale points to fit frame
                scaled_points = [scale_point(p) for p in points]
                
                # Get color for this direction
                color = direction_to_color.get(direction, (0, 0, 0))
                
                # Draw the trajectory line with higher thickness
                for i in range(len(scaled_points) - 1):
                    cv2.line(visualization, 
                             scaled_points[i], 
                             scaled_points[i+1], 
                             color, 
                             2,  # Thicker line
                             cv2.LINE_AA)  # Anti-aliased line
                
                # Draw a larger circle at the starting point
                cv2.circle(visualization, 
                          scaled_points[0], 
                          5,  # Larger radius
                          color, 
                          -1)  # Filled circle
                
                # Draw an arrow at the end point
                end_pt = scaled_points[-1]
                prev_pt = scaled_points[-2] if len(scaled_points) > 1 else scaled_points[0]
                
                # Calculate arrow direction
                dx = end_pt[0] - prev_pt[0]
                dy = end_pt[1] - prev_pt[1]
                
                # Skip if the line is too short
                if dx**2 + dy**2 > 100:  # Min squared distance
                    # Calculate angle
                    angle = np.arctan2(dy, dx)
                    
                    # Arrow parameters
                    arrow_length = 15
                    arrow_angle = np.pi/6  # 30 degrees
                    
                    # Calculate arrow points
                    pt1_x = end_pt[0] - arrow_length * np.cos(angle + arrow_angle)
                    pt1_y = end_pt[1] - arrow_length * np.sin(angle + arrow_angle)
                    pt2_x = end_pt[0] - arrow_length * np.cos(angle - arrow_angle)
                    pt2_y = end_pt[1] - arrow_length * np.sin(angle - arrow_angle)
                    
                    # Draw arrow head
                    cv2.line(visualization, end_pt, (int(pt1_x), int(pt1_y)), color, 2, cv2.LINE_AA)
                    cv2.line(visualization, end_pt, (int(pt2_x), int(pt2_y)), color, 2, cv2.LINE_AA)
    
    # Draw direction legends with larger, clearer text
    legend_height = 30
    legend_spacing = 40
    legend_x = 20
    legend_y = 30
    
    # Direction counts for statistics
    direction_counts = Counter(df['direction'])
    
    # Add semi-transparent background for legends
    overlay = visualization.copy()
    legend_bg_height = legend_spacing * (len(direction_to_color) + 1)
    cv2.rectangle(overlay, 
                 (10, 10), 
                 (300, legend_bg_height + 20), 
                 (255, 255, 255), 
                 -1)
    cv2.addWeighted(overlay, 0.7, visualization, 0.3, 0, visualization)
    
    # Draw title
    cv2.putText(visualization, 
                f"Vehicle Trajectories: {video_id}", 
                (legend_x, legend_y), 
                cv2.FONT_HERSHEY_SIMPLEX, 
                0.7, 
                (0, 0, 0), 
                2, 
                cv2.LINE_AA)
    
    # Draw direction legends
    for i, (direction, color) in enumerate(direction_to_color.items()):
        # Find corresponding angle and count
        cluster_idx = next((idx for idx, name in cluster_to_name.items() 
                           if name == direction), None)
        
        if cluster_idx is not None:
            angle = cluster_angles[cluster_idx]
            count = direction_counts.get(direction, 0)
            percentage = (count / len(df)) * 100 if len(df) > 0 else 0
            
            # Draw colored rectangle for direction
            cv2.rectangle(visualization, 
                         (legend_x, legend_y + legend_spacing * (i+1) - 15), 
                         (legend_x + 20, legend_y + legend_spacing * (i+1) + 5), 
                         color, 
                         -1)
            
            # Draw direction text
            text = f"{direction}: {angle:.1f}° ({count} vehicles, {percentage:.1f}%)"
            cv2.putText(visualization, 
                        text, 
                        (legend_x + 30, legend_y + legend_spacing * (i+1)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 
                        0.6, 
                        (0, 0, 0), 
                        1, 
                        cv2.LINE_AA)
    
    # Draw major direction arrows from center
    # This will help make the pattern more obvious
    arrow_length = min(frame_width, frame_height) * 0.2
    
    for cluster_idx, angle_deg in enumerate(cluster_angles):
        direction_name = cluster_to_name[cluster_idx]
        count = direction_counts.get(direction_name, 0)
        
        # Skip directions with very few vehicles
        if count < 2:
            continue
            
        # Calculate percentage of vehicles in this direction
        percentage = count / len(df) * 100 if len(df) > 0 else 0
        
        # Convert to radians
        angle_rad = np.radians(angle_deg)
        
        # Calculate arrow endpoint
        end_x = int(center_x + arrow_length * np.cos(angle_rad))
        end_y = int(center_y + arrow_length * np.sin(angle_rad))
        
        # Get color
        color = direction_to_color[direction_name]
        
        # Scale thickness based on percentage (min 3, max 12)
        thickness = max(3, min(12, int(percentage / 5)))
        
        # Draw main direction arrow
        cv2.arrowedLine(visualization, 
                       (center_x, center_y), 
                       (end_x, end_y), 
                       color, 
                       thickness, 
                       cv2.LINE_AA, 
                       tipLength=0.2)
    
    # Save visualization
    output_path = f"{base_filename.replace('_manual', '')}_improved_trajectories.png"
    cv2.imwrite(output_path, visualization)
    
    # Also save a version with better contrast if the original frame is dark
    gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    avg_brightness = np.mean(gray)
    
    if avg_brightness < 100:  # If original frame is dark
        # Create a brightened version
        bright_viz = visualization.copy()
        
        # Add a semi-transparent white overlay to improve contrast
        overlay = np.ones_like(bright_viz) * 255
        cv2.addWeighted(overlay, 0.3, bright_viz, 0.7, 0, bright_viz)
        
        # Re-draw all trajectories with higher contrast
        # (Code similar to above but with brighter colors)
        
        # Save high-contrast version
        bright_output_path = f"{base_filename.replace('_manual', '')}_bright_trajectories.png"
        cv2.imwrite(bright_output_path, bright_viz)
        print(f"Also saved high-contrast version: {bright_output_path}")
    
    print(f"Visualization saved to: {output_path}")
    return output_path